# ToxAgent Pro — Agentic AI for Animal-Free Toxicology
### A Complete Autonomous Pipeline: 25 Tools · 8 Agent Nodes · LangGraph State Machine

**Author:** Himanshu Goel | [hgoelgithub.github.io](https://hgoelgithub.github.io)

---

## What ToxAgent Pro does

ToxAgent Pro is an autonomous multi-step AI agent that performs **every major toxicology assessment** needed to replace animal experiments for a given compound. You give it a SMILES string. It returns a complete regulatory-grade safety dossier.

```
                        ┌─────────────────────────────────┐
   SMILES input         │         TOXAGENT PRO             │     Regulatory
   ──────────────►      │                                   │ ──► Dossier
                        │  ┌─────────────────────────────┐ │
                        │  │     SUPERVISOR AGENT         │ │
                        │  │  (routes, coordinates, WoE) │ │
                        │  └──────┬──────────────────────┘ │
                        │         │ dispatches to            │
                        │  ┌──────▼──────────────────────┐ │
                        │  │    8 SPECIALIST AGENTS       │ │
                        │  │                               │ │
                        │  │  ① ADMET       ② Geno        │ │
                        │  │  ③ Cardiac     ④ DILI        │ │
                        │  │  ⑤ OoC/Org     ⑥ PBPK-IVIVE │ │
                        │  │  ⑦ AOP         ⑧ Literature  │ │
                        │  └──────────────────────────────┘ │
                        │                                   │
                        │  25 TOOLS (all animal-free)      │
                        │  Memory · HITL · Uncertainty      │
                        └─────────────────────────────────┘
```

## The 25 tools cover every animal test replacement

| # | Tool | Replaces animal test | Guideline |
|---|------|---------------------|-----------|
| 1 | `admet_profile` | Physicochemical screening | Ro5, Veber |
| 2 | `structural_alerts_ich_m7` | Genotox Ames screen | ICH M7(R2) |
| 3 | `qsar_ames` | Ames mutagenicity | ICH M7(R2) |
| 4 | `qsar_ld50` | Acute oral LD50 | OECD 423, GHS |
| 5 | `herg_ic50` | hERG patch-clamp | ICH S7B |
| 6 | `cipa_multichannel` | Multi-channel cardiac | ICH E14/S7B 2022 |
| 7 | `hipscm_mea` | In vivo QT study | CiPA Tier 2 |
| 8 | `dili_prediction` | Hepatotox rat study | DILIrank |
| 9 | `liver_organoid` | 28-day liver tox | InSphero/CN Bio |
| 10 | `ooc_liver` | Repeat-dose hepatotox | OoC liver |
| 11 | `ooc_kidney` | Nephrotox rat study | OoC kidney |
| 12 | `skin_sensitisation_2o3` | LLNA mouse | OECD TG 497 |
| 13 | `bbb_penetration` | CNS animal studies | CNS MPO |
| 14 | `neurotox_screen` | DNT Tier 1 | ICH S5(R3) |
| 15 | `pbpk_ivive` | Dose range finding | EPA HTTK |
| 16 | `population_pbpk` | Animal dose-finding | Monte Carlo PBPK |
| 17 | `cytotox_correction` | Cytotox artefact removal | EPA burst ratio |
| 18 | `pains_filter` | False-positive HTS | Baell & Holloway 2010 |
| 19 | `genotox_battery` | Full ICH S2 battery | Ames+MN+CA in silico |
| 20 | `reactive_metabolite` | Reactive met. assay | Protein binding |
| 21 | `endocrine_disruption` | ER/AR binding assay | OECD TG 455/457 |
| 22 | `aop_scoring` | Mechanism-based studies | OECD AOP-Wiki |
| 23 | `read_across` | Data-gap filling | OECD RAAF |
| 24 | `literature_search` | Expert review | PubMed/ChEMBL |
| 25 | `regulatory_classification` | Final WoE summary | OECD GD 255 |

## Notebook structure

| Section | Content |
|---------|---------|
| 1 | Dependencies & imports |
| 2 | All 25 animal-free tools |
| 3 | Eight specialist agents |
| 4 | LangGraph state machine |
| 5 | Supervisor + routing |
| 6 | Memory (episodic + semantic) |
| 7 | Human-in-the-loop (HITL) |
| 8 | Full pipeline execution |
| 9 | Visualisation dashboard |
| 10 | Regulatory dossier output |

---
## Section 1 — Dependencies & Setup

In [ ]:
# ── Install ───────────────────────────────────────────────────────────────────
# !pip install rdkit scikit-learn xgboost shap matplotlib seaborn pandas scipy

import os, re, json, time, warnings, textwrap
from dataclasses import dataclass, field
from typing import Any, Literal, Optional
from collections import defaultdict, deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
warnings.filterwarnings('ignore')

from rdkit import Chem, DataStructs
from rdkit.Chem import (Descriptors, rdMolDescriptors, AllChem,
                         QED, FilterCatalog)
from rdkit.Chem.FilterCatalog import FilterCatalogParams
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem.MolStandardize import rdMolStandardize
from scipy import stats, optimize
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score
from statsmodels.stats.multitest import multipletests

# ── Global configuration ───────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# LLM simulation flag (set to False and provide API key for real LLM)
SIMULATE_LLM = True

# Regulatory thresholds (ICH / OECD / FDA)
THRESHOLDS = {
    "ICH_M7_sensitivity":     0.90,   # minimum model sensitivity
    "TTC_class2_ug_day":      1.5,    # ICH M7 threshold of toxicological concern
    "TTC_class1_ug_day":      0.0025, # ICH M7 Class 1 (potent mutagen)
    "hERG_flag_uM":           1.0,    # hERG IC50 concern threshold
    "dFPDc_high_ms":          30.0,   # CiPA Tier 2 HIGH risk
    "dFPDc_mod_ms":           20.0,   # CiPA Tier 2 MODERATE
    "DILI_ATP_pct":           70.0,   # ATP depletion DILI threshold
    "skin_dpra_pct":          6.38,   # DPRA mean depletion (OECD TG 442C)
    "power_min":              0.80,   # minimum statistical power (3Rs)
    "AD_tanimoto_min":        0.40,   # applicability domain minimum Tc
}

print("ToxAgent Pro configuration loaded ✓")
print(f"LLM mode: {'Simulated (tutorial)' if SIMULATE_LLM else 'Real (API)'}")
print(f"Thresholds: {len(THRESHOLDS)} regulatory parameters loaded")

---
## Section 2 — The 25 Animal-Free Tools

Each tool is a pure Python function. The agent decides which tools to call, in what order, with what arguments. Tools return structured JSON — parseable by both humans and LLMs.

In [ ]:
# ── Tool registry infrastructure ──────────────────────────────────────────────
@dataclass
class ToxTool:
    name:        str
    description: str    # what the LLM reads to decide when to use this tool
    endpoint:    str    # toxicology endpoint
    replaces:    str    # which animal test it replaces
    guideline:   str    # regulatory guideline
    func:        Any    # callable

    def run(self, **kwargs) -> str:
        try:
            result = self.func(**kwargs)
            return json.dumps(result, indent=2, default=str)
        except Exception as e:
            return json.dumps({"error": str(e), "tool": self.name})

# ── Helper functions ───────────────────────────────────────────────────────────
def _mol(smiles: str):
    mol = Chem.MolFromSmiles(smiles.strip()) if smiles else None
    if mol is None:
        raise ValueError(f"Invalid SMILES: {smiles!r}")
    return mol

def _fp(smiles: str):
    mol = _mol(smiles)
    return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048), dtype=np.uint8)

def _desc(mol) -> dict:
    return {
        "mw":    Descriptors.MolWt(mol),
        "logp":  Descriptors.MolLogP(mol),
        "tpsa":  Descriptors.TPSA(mol),
        "hbd":   rdMolDescriptors.CalcNumHBD(mol),
        "hba":   rdMolDescriptors.CalcNumHBA(mol),
        "rb":    rdMolDescriptors.CalcNumRotatableBonds(mol),
        "naro":  rdMolDescriptors.CalcNumAromaticRings(mol),
        "fsp3":  rdMolDescriptors.CalcFractionCSP3(mol),
        "qed":   QED.qed(mol),
    }

print("Tool infrastructure ready ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TOOLS 1-5: ADMET, Structural Alerts, QSAR (Ames + LD50), hERG
# ══════════════════════════════════════════════════════════════════════════════

def tool_admet_profile(smiles: str) -> dict:
    """Compute full ADMET physicochemical profile. Returns Ro5, Veber, CNS MPO."""    mol = _mol(smiles)
    d   = _desc(mol)
    ro5_viol = sum([d["mw"]>500, d["logp"]>5, d["hbd"]>5, d["hba"]>10])
    veber    = d["tpsa"] <= 140 and d["rb"] <= 10
    cns_mpo  = sum([d["mw"]<=360, 1<=d["logp"]<=3, d["tpsa"]<=90,
                    d["hbd"]<=1, d["logp"]<5, d["naro"]<=2])
    return {
        "MW": round(d["mw"],2), "LogP": round(d["logp"],3),
        "TPSA": round(d["tpsa"],1), "HBD": d["hbd"], "HBA": d["hba"],
        "RotBonds": d["rb"], "ArRings": d["naro"],
        "Fsp3": round(d["fsp3"],3), "QED": round(d["qed"],4),
        "Formula": rdMolDescriptors.CalcMolFormula(mol),
        "Ro5_violations": ro5_viol,
        "oral_bioavailability": "Likely" if ro5_viol<=1 else "Poor",
        "Veber_oral": veber,
        "CNS_MPO_score": f"{cns_mpo}/6",
        "CNS_penetrant": cns_mpo >= 4,
        "lead_like": d["mw"]<=450 and d["logp"]<=4 and d["rb"]<=7,
    }

# ICH M7 structural alerts (21 patterns across Classes 1-3)
_ICH_ALERTS = {
    "Nitrosamine":           "[N;!$(N=O)]-N=O",
    "N_Nitroso":             "[#6][N;H0]([#6])N=O",
    "Diazonium":             "[#6][N+]#N",
    "Aromatic_nitro":        "c[N+](=O)[O-]",
    "Aliphatic_nitro":       "C[N+](=O)[O-]",
    "Aromatic_primary_amine":"[NH2]c",
    "Epoxide":               "[C;R0]1OC1",
    "Aziridine":             "C1CN1",
    "Michael_acceptor":      "C=CC=O",
    "Alpha_beta_unsat_ald":  "C=CC=O",
    "Acyl_halide":           "C(=O)[Cl,Br]",
    "Isocyanate":            "N=C=O",
    "Hydrazine":             "[NH2]N",
    "Hydroxamic_acid":       "C(=O)NO",
    "Sultone":               "C1CCS(=O)(=O)O1",
    "Propiolactone":         "C1CC(=O)O1",
    "Furan_metabolite":      "c1ccoc1",
    "Thiophene_metabolite":  "c1ccsc1",
    "Alkenyl_halide":        "C=C[Cl,Br,I]",
    "Aromatic_azo":          "c-N=N-c",
    "Primary_alkyl_halide":  "[Br,I][CX4]",
}

def tool_structural_alerts(smiles: str) -> dict:
    """ICH M7(R2) two-method framework — structural alert screening (Method 1)."""    mol  = _mol(smiles)
    hits = {}
    for name, smarts in _ICH_ALERTS.items():
        patt = Chem.MolFromSmarts(smarts)
        if patt and mol.HasSubstructMatch(patt):
            hits[name] = True
    c1 = {k for k in hits if "Nitroso" in k or "Diazonium" in k}
    c2 = {k for k in hits if k not in c1}
    if c1:   ich_class = "Class 1 — Known human mutagen (AVOID)"
    elif c2: ich_class = "Class 2/3 — Structural concern (TTC applies)"
    else:    ich_class = "Class 5 — No structural alert"
    return {
        "n_alerts": len(hits), "alerts": list(hits.keys()),
        "ich_m7_class": ich_class, "sa_call": "POSITIVE" if hits else "NEGATIVE",
        "ttc_ug_day": THRESHOLDS["TTC_class1_ug_day"] if c1 else
                      (THRESHOLDS["TTC_class2_ug_day"] if c2 else "No TTC limit"),
    }

def tool_qsar_ames(smiles: str) -> dict:
    """QSAR Ames mutagenicity prediction (ICH M7 Method 2). RF + ECFP4."""    np.random.seed(SEED)
    mol  = _mol(smiles)
    d    = _desc(mol)
    has_nitro = mol.HasSubstructMatch(Chem.MolFromSmarts("[N+](=O)[O-]"))
    has_ar_am = mol.HasSubstructMatch(Chem.MolFromSmarts("[NH2]c"))
    # Rule-based QSAR proxy (replace with trained RF in production)
    prob = 0.08 + 0.50*has_nitro + 0.35*has_ar_am + 0.10*(d["logp"]>3)
    prob = float(np.clip(prob + np.random.normal(0, 0.05), 0.01, 0.99))
    in_ad = d["mw"] < 800 and d["logp"] < 7
    return {
        "ames_probability": round(prob, 3),
        "qsar_call":   "POSITIVE" if prob >= 0.5 else "NEGATIVE",
        "confidence":  "HIGH" if in_ad else "LOW (outside AD)",
        "within_AD":   in_ad,
        "model":       "RF-ECFP4 (simulated — replace with trained model)",
        "sensitivity": 0.92, "specificity": 0.78,  # literature values
        "guideline":   "ICH M7(R2) 2023",
    }

def tool_qsar_ld50(smiles: str) -> dict:
    """Predict acute oral LD50 (rat) and GHS category. Replaces OECD 423."""    mol  = _mol(smiles)
    d    = _desc(mol)
    # Simplified QSAR (Zhu 2009 approach)
    log_ld50 = 3.5 - 0.2*d["logp"] + 0.001*d["mw"] - 0.01*d["tpsa"]
    log_ld50 = float(np.clip(log_ld50 + np.random.normal(0, 0.3), 0.5, 5.0))
    ld50     = round(10**log_ld50, 0)
    if   ld50 <=  5:   ghs = "Category 1 (Fatal)"
    elif ld50 <= 50:   ghs = "Category 2 (Fatal)"
    elif ld50 <= 300:  ghs = "Category 3 (Toxic)"
    elif ld50 <= 2000: ghs = "Category 4 (Harmful)"
    else:              ghs = "Category 5 / Unclassified"
    return {
        "LD50_mg_kg_rat_oral": ld50,
        "log_LD50": round(log_ld50, 2),
        "GHS_category": ghs,
        "replaces":  "OECD TG 423 Fixed Dose / TG 425 Up-and-Down",
        "guideline": "OECD GD 69, EPA QSAR",
        "3Rs_saving": "~10 rats per LD50 test avoided",
    }

def tool_herg_ic50(smiles: str) -> dict:
    """Predict hERG IC50 (cardiac safety). Replaces hERG patch-clamp animal."""    mol  = _mol(smiles)
    d    = _desc(mol)
    basic_n = sum(1 for a in mol.GetAtoms()
                  if a.GetAtomicNum()==7 and a.GetTotalNumHs()>0)
    risk_score = 0.30*max(0,d["logp"]-2) + 0.002*max(0,d["mw"]-300)                + 0.15*basic_n + 0.10*d["naro"]
    ic50 = 10**(-risk_score + 2.0 + np.random.normal(0, 0.3))
    ic50 = float(np.clip(ic50, 0.001, 10000))
    risk = ("HIGH"   if ic50 < THRESHOLDS["hERG_flag_uM"]
       else "MEDIUM" if ic50 < 10.0
       else "LOW")
    return {
        "hERG_IC50_uM": round(ic50, 3),
        "risk_level":   risk,
        "safety_margin_needed": "IC50 > 30× free Cmax (ICH S7B)",
        "flag":         ic50 < THRESHOLDS["hERG_flag_uM"],
        "recommendation": "Proceed to CiPA multi-channel assay" if risk=="MEDIUM"
                          else ("hERG concern — CiPA + hiPS-CM REQUIRED" if risk=="HIGH"
                                else "Acceptable — monitoring sufficient"),
    }

print("Tools 1-5 defined ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TOOLS 6-10: CiPA, hiPS-CM MEA, DILI, Liver Organoid, OoC Liver
# ══════════════════════════════════════════════════════════════════════════════

def tool_cipa_multichannel(smiles: str) -> dict:
    """CiPA 7-channel ion current assessment. Replaces hERG-only per ICH E14/S7B."""    np.random.seed(SEED + abs(hash(smiles)) % 100)
    mol = _mol(smiles)
    d   = _desc(mol)
    basic_n = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==7 and a.GetTotalNumHs()>0)
    herg_base = max(0.001, 10**(-(0.25*d["logp"] + 0.12*basic_n + 0.07*d["naro"] - 1.5)
                                 + np.random.normal(0, 0.4)))
    channels = {
        "IKr_hERG":  herg_base,
        "INaL_Nav":  herg_base * np.random.uniform(2,8),
        "ICaL_Cav":  herg_base * np.random.uniform(5,20),
        "IKs_KCNQ1": herg_base * np.random.uniform(3,15),
        "INaF_Nav":  herg_base * np.random.uniform(1,5),
        "If_HCN4":   herg_base * np.random.uniform(10,100),
        "Ito_Kv4":   herg_base * np.random.uniform(8,50),
    }
    directions = {"IKr_hERG":1,"INaL_Nav":1,"ICaL_Cav":-1,"IKs_KCNQ1":1,
                  "INaF_Nav":1,"If_HCN4":0,"Ito_Kv4":0}
    weights   = {"IKr_hERG":0.40,"INaL_Nav":0.15,"ICaL_Cav":0.20,
                 "IKs_KCNQ1":0.10,"INaF_Nav":0.05,"If_HCN4":0.05,"Ito_Kv4":0.05}
    cmax = max(0.001, herg_base * 0.1)
    net_risk = sum(weights[ch] * (1/(1+ic50/cmax)) * directions[ch]
                   for ch, ic50 in channels.items())
    tdp = ("HIGH TdP" if net_risk>0.15 else "INTERMEDIATE" if net_risk>0.05 else "LOW TdP")
    return {
        "IC50_uM": {ch: round(float(ic50),3) for ch, ic50 in channels.items()},
        "CiPA_net_score": round(float(net_risk), 4),
        "TdP_category":   tdp,
        "replaces": "hERG-only ICH S7B test",
        "guideline": "ICH E14/S7B 2022, CiPA Initiative",
    }

def tool_hipscm_mea(smiles: str, concentration_uM: float = 1.0) -> dict:
    """hiPS-CM MEA electrophysiology — CiPA Tier 2. Replaces rabbit in vivo QT."""    np.random.seed(SEED + int(concentration_uM*10) % 100)
    mol = _mol(smiles)
    d   = _desc(mol)
    basic_n  = sum(1 for a in mol.GetAtoms() if a.GetAtomicNum()==7 and a.GetTotalNumHs()>0)
    herg_act = np.clip(0.2*d["logp"] + 0.15*basic_n + 0.05*d["naro"] - 0.4, 0, 0.9)
    fpd_base = np.random.normal(380, 12)
    fpd_drug = fpd_base * (1 + 0.45*herg_act)
    bpm_base = np.random.normal(58, 4)
    bpm_drug = bpm_base * (1 - 0.28*herg_act)
    rr_drug  = 60000 / bpm_drug
    fpdc_base= fpd_base / np.sqrt((60000/bpm_base)/1000)
    fpdc_drug= fpd_drug / np.sqrt(rr_drug/1000)
    delta    = fpdc_drug - fpdc_base
    ead      = np.random.rand() < herg_act**2 * 1.8
    risk     = ("HIGH"     if delta > THRESHOLDS["dFPDc_high_ms"] or ead
           else "MODERATE" if delta > THRESHOLDS["dFPDc_mod_ms"]
           else "LOW")
    return {
        "FPD_baseline_ms": round(fpd_base,1), "FPD_drug_ms": round(fpd_drug,1),
        "delta_FPDc_ms":   round(float(delta),1),
        "BPM_baseline":    round(bpm_base,1), "BPM_drug": round(bpm_drug,1),
        "EAD_observed":    bool(ead), "risk_tier": risk,
        "cipa_tier2_call": risk,
        "replaces": "Rabbit in vivo QT study (ICH S7B)",
        "guideline": "ICH E14/S7B 2022, CiPA Tier 2",
    }

def tool_dili_prediction(smiles: str) -> dict:
    """Predict DILI severity using DILIrank ML model. Replaces rat hepatotox study."""    np.random.seed(SEED + abs(hash(smiles)) % 50)
    mol = _mol(smiles)
    d   = _desc(mol)
    mito   = mol.HasSubstructMatch(Chem.MolFromSmarts("[CX4][N+](C)(C)C"))
    bsep   = d["mw"] > 400 and d["logp"] > 3
    rm_risk= mol.HasSubstructMatch(Chem.MolFromSmarts("c1ccoc1")) or              mol.HasSubstructMatch(Chem.MolFromSmarts("C=CC=O"))
    dili_score = (0.25*(d["logp"]>3) + 0.20*(d["mw"]>350) + 0.20*bsep
                + 0.20*rm_risk + 0.15*mito)
    dili_score += np.random.normal(0, 0.08)
    dili_score  = np.clip(dili_score, 0, 1)
    dili_class  = ("Most-concern"    if dili_score > 0.65
              else "Less-concern"    if dili_score > 0.35
              else "No-concern")
    return {
        "DILI_score":      round(float(dili_score), 3),
        "DILIrank_class":  dili_class,
        "mechanism_flags": {
            "mitochondrial_liability": bool(mito),
            "BSEP_concern":            bool(bsep),
            "reactive_metabolite":     bool(rm_risk),
        },
        "replaces": "28-day rat hepatotoxicity study",
        "guideline": "DILIrank (Chen 2016), FDA DILI guidance",
    }

def tool_liver_organoid(smiles: str, concentration_uM: float = 10.0,
                         days: int = 14) -> dict:
    """3D liver organoid (iPSC spheroid) DILI assay. Replaces in vivo hepatotox."""    np.random.seed(SEED + int(concentration_uM) % 77)
    mol = _mol(smiles)
    d   = _desc(mol)
    tox = np.clip((d["logp"]/5 + d["naro"]/4) * (np.log10(concentration_uM+1)/3), 0, 1)
    tox += np.random.normal(0, 0.06)
    atp   = max(5,  round(100*(1-0.80*tox) + np.random.normal(0,3), 1))
    alb   = max(0,  round(20 *(1-0.70*tox) + np.random.normal(0,1.5), 2))
    alt   = max(0,  round(8  * tox*15      + np.random.normal(0,2), 1))
    ros   = max(1,  round(1  + 4*tox       + np.random.normal(0,0.3), 2))
    steat = max(0,  round(30 * tox         + np.random.normal(0,3), 1))
    gsh   = max(0,  round(70 * tox         + np.random.normal(0,5), 1))
    concerns = []
    if atp   < THRESHOLDS["DILI_ATP_pct"]: concerns.append("ATP depletion")
    if alt   > 30:   concerns.append("ALT elevation")
    if ros   > 2.0:  concerns.append("ROS excess")
    if steat > 15:   concerns.append("Steatosis")
    if gsh   > 30:   concerns.append("GSH depletion")
    dili_risk = ("HIGH" if len(concerns)>=3 else "MODERATE" if len(concerns)>=1 else "NONE")
    return {
        "ATP_pct": atp, "albumin_ug_mL_d": alb, "ALT_U_L": alt,
        "ROS_fold": ros, "steatosis_pct": steat, "GSH_depletion_pct": gsh,
        "DILI_risk": dili_risk, "n_concerns": len(concerns), "concerns": concerns,
        "exposure_days": days, "concentration_uM": concentration_uM,
        "replaces": "28-day rat liver toxicity study (OECD TG 407)",
        "platform": "InSphero GravityPLUS / CN Bio PhysioMimix",
    }

def tool_ooc_liver(smiles: str, concentration_uM: float = 10.0) -> dict:
    """Liver organ-on-chip (microphysiological system). Replaces repeat-dose study."""    np.random.seed(SEED + int(concentration_uM*3) % 88)
    mol = _mol(smiles)
    d   = _desc(mol)
    tox = np.clip(d["logp"]/6 * np.log10(concentration_uM+1)/3, 0, 1)
    tox += np.random.normal(0, 0.05)
    endpoints = {
        "TEER_ohm_cm2":      max(10,  round(250*(1-0.8*tox)  + np.random.normal(0,10), 0)),
        "albumin_ug_mL_d":   max(0,   round(15 *(1-0.7*tox)  + np.random.normal(0,1.2), 2)),
        "CYP3A4_pct":        max(0,   round(100*(1-0.85*tox) + np.random.normal(0,6), 1)),
        "urea_ug_mL_d":      max(0,   round(12 *(1-0.65*tox) + np.random.normal(0,1), 2)),
        "bile_flux_relative":max(0,   round(1  *(1-0.7*tox)  + np.random.normal(0,0.1), 3)),
    }
    n_tox = sum([
        endpoints["TEER_ohm_cm2"]    < 200,
        endpoints["albumin_ug_mL_d"] < 10,
        endpoints["CYP3A4_pct"]      < 70,
        endpoints["urea_ug_mL_d"]    < 8,
    ])
    severity = ("SEVERE" if n_tox>=3 else "MODERATE" if n_tox>=2 else "MILD" if n_tox>=1 else "NONE")
    return {**endpoints, "severity": severity, "n_toxic_endpoints": n_tox,
            "replaces": "Repeat-dose hepatotoxicity (OECD TG 408)",
            "platform": "Emulate Bio / CN Bio PhysioMimix"}

print("Tools 6-10 defined ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TOOLS 11-17: OoC Kidney, Skin Sens, BBB, Neurotox, PBPK, PopPBPK, CytotoxCorr
# ══════════════════════════════════════════════════════════════════════════════

def tool_ooc_kidney(smiles: str, concentration_uM: float = 10.0) -> dict:
    """Kidney organ-on-chip (proximal tubule). Replaces nephrotox animal study."""    np.random.seed(SEED + 11)
    mol = _mol(smiles); d = _desc(mol)
    tox = np.clip((d["logp"]/5) * np.log10(concentration_uM+1)/3, 0, 1)
    tox += np.random.normal(0, 0.05)
    return {
        "KIM1_ng_mL":        max(0, round(0.5 + 8*tox + np.random.normal(0,0.5), 2)),
        "NGAL_ng_mL":        max(0, round(2.0 + 15*tox+ np.random.normal(0,1), 2)),
        "creatinine_clear_rel": max(0, round(1 - 0.7*tox + np.random.normal(0,0.06), 3)),
        "cell_viability_pct":max(5,  round(100*(1-0.75*tox)+np.random.normal(0,4), 1)),
        "nephrotox_call": "CONCERN" if tox > 0.35 else "NO CONCERN",
        "replaces": "Rat nephrotoxicity study (OECD TG 407, ICH S7A)",
        "platform": "Mimetas OrganoPlate / Emulate Kidney Chip",
    }

def tool_skin_sensitisation(smiles: str) -> dict:
    """OECD TG 497 Defined Approach 2o3. Replaces LLNA/GPMT mouse/guinea pig."""    np.random.seed(SEED + 5)
    mol = _mol(smiles); d = _desc(mol)
    michael = mol.HasSubstructMatch(Chem.MolFromSmarts("[$(C=CC=O),$(C=CS)]"))
    ald     = mol.HasSubstructMatch(Chem.MolFromSmarts("[CHO]"))
    epox    = mol.HasSubstructMatch(Chem.MolFromSmarts("C1OC1"))
    react   = 0.3*michael + 0.25*ald + 0.2*epox
    react  += np.random.normal(0, 0.05)
    dpra_dep = float(np.clip(react*80, 0, 100))
    ks_imax  = float(np.clip(react*200, 10, 300))
    hclat_ec3= float(np.clip(100*(1-react), 0.5, 100))
    dpra_pos = dpra_dep >= THRESHOLDS["skin_dpra_pct"]
    ks_pos   = ks_imax  >= 150
    hcl_pos  = hclat_ec3 < 50
    n_pos    = sum([dpra_pos, ks_pos, hcl_pos])
    return {
        "DPRA_depletion_pct": round(dpra_dep,1), "DPRA_call": "POS" if dpra_pos else "NEG",
        "KeratinoSens_imax": round(ks_imax,1),  "KS_call":   "POS" if ks_pos else "NEG",
        "hCLAT_EC3_ug_mL":  round(hclat_ec3,1),"hCLAT_call":"POS" if hcl_pos else "NEG",
        "n_positive": n_pos,
        "hazard_call": "SENSITISER" if n_pos>=2 else "NON-SENSITISER",
        "replaces": "LLNA (OECD TG 429) / GPMT (OECD TG 406) — ~26 mice avoided",
        "guideline": "OECD TG 442C/D/E + TG 497 DA2 (2023)",
    }

def tool_bbb_penetration(smiles: str) -> dict:
    """BBB penetration scoring. Replaces CNS animal distribution study."""    mol = _mol(smiles); d = _desc(mol)
    # BBB-Score (Gupta 2019) + CNS MPO (Wager 2010)
    bbb_score = sum([d["mw"]<=400, d["logp"]<=3, d["tpsa"]<=80,
                     d["hbd"]<=1, d["naro"]<=2]) / 5.0
    cns_mpo   = sum([d["mw"]<=360, 1<=d["logp"]<=3, d["tpsa"]<=90,
                     d["hbd"]<=1, d["logp"]<5, d["naro"]<=2])
    kp_uu_brain = float(np.clip(10**(bbb_score*2 - 1.5), 0.001, 5))
    return {
        "BBB_score_0to1":    round(bbb_score, 3),
        "CNS_MPO_6scale":    cns_mpo,
        "Kp_uu_brain_est":   round(kp_uu_brain, 4),
        "CNS_penetrant":     cns_mpo >= 4 and bbb_score >= 0.6,
        "P_gp_substrate":    d["mw"]>400 and d["hbd"]>2,
        "replaces": "IV brain distribution / CSF sampling (rats)",
    }

def tool_neurotox_screen(smiles: str) -> dict:
    """Developmental neurotox (DNT) Tier 1 screen. Replaces rat DNT study."""    np.random.seed(SEED + 14)
    mol = _mol(smiles); d = _desc(mol)
    bbb = tool_bbb_penetration(smiles)["CNS_penetrant"]
    ache = mol.HasSubstructMatch(Chem.MolFromSmarts("[P](=O)([O-,OH])[O,S]"))
    nmda = d["logp"] > 2 and d["mw"] < 400 and bbb
    excit_tox = float(np.clip(0.3*int(nmda) + 0.4*int(ache) + 0.1*d["logp"]/5, 0, 1))
    excit_tox += np.random.normal(0, 0.05)
    return {
        "BBB_penetrant":       bbb,
        "AChE_inhibitor_flag": bool(ache),
        "NMDA_concern":        bool(nmda),
        "excitotox_score":     round(float(excit_tox), 3),
        "DNT_tier1_call":      "CONCERN" if excit_tox>0.4 or ache else "NO CONCERN",
        "MEA_network_flag":    excit_tox > 0.3,
        "replaces": "Rat/rabbit DNT study (OECD TG 426, ICH S5)",
        "recommendation": "Proceed to cerebral organoid MEA" if excit_tox>0.3 else "No further testing",
    }

def tool_pbpk_ivive(smiles: str, dose_mg_kg: float = 1.0) -> dict:
    """EPA HTTK IVIVE: in vitro EC50 → human AED. Replaces dose-range finding."""    np.random.seed(SEED + 4)
    mol = _mol(smiles); d = _desc(mol)
    mw   = d["mw"]
    logp = d["logp"]
    fup  = float(np.clip(10**(-0.028*logp - 0.0038*mw + 1.2), 0.001, 1.0))
    clint= float(np.clip(10**(0.35*logp + 0.15*d["naro"] - 0.002*mw)
                          + np.random.normal(0, 0.3), 0.1, 1000))
    Qh   = 90.0; mppgl = 45.0; liver_g = 1500.0
    cl_liver = clint * mppgl * liver_g / 1000
    CLh  = (Qh * cl_liver * fup) / (Qh + cl_liver * fup)
    Vd   = max(0.5, 0.2 + 0.8*logp) * 70
    t_half = round(0.693 * Vd / CLh, 2) if CLh > 0 else 99
    css  = (dose_mg_kg * 70 * 1000 / mw / 1440) / (CLh/1000) if CLh>0 else 99
    aed  = round(float(css * CLh / 1000 * mw * 1440 / (70*1000)), 5)
    return {
        "fup": round(fup,4), "CLint_mL_min_mg": round(clint,2),
        "CLh_L_h": round(float(CLh),2), "Vd_L": round(float(Vd),1),
        "Css_uM": round(float(css),4), "t_half_h": t_half,
        "AED_mg_kg_day": aed,
        "TTC_ug_day": round(aed*1000*70*1000, 2),
        "TTC_concern": aed*1000*70*1000 < THRESHOLDS["TTC_class2_ug_day"],
        "replaces": "Animal dose range-finding + TK study",
        "guideline": "EPA HTTK, Rotroff 2010, Wetmore 2012",
    }

def tool_population_pbpk(smiles: str, dose_mg_kg: float = 1.0,
                           n_subjects: int = 200) -> dict:
    """Monte Carlo population PBPK. Replaces animal TK study + dose-finding."""    np.random.seed(SEED)
    mol = _mol(smiles); d = _desc(mol)
    bw_pop    = np.random.lognormal(np.log(70), 0.20, n_subjects)
    clint_pop = np.random.lognormal(np.log(max(0.5,d["logp"])*8), 0.40, n_subjects)
    fup_pop   = np.clip(np.random.lognormal(np.log(0.12), 0.35, n_subjects), 0.002, 1)
    Cmaxs     = []
    for i in range(n_subjects):
        bw, clint, fup = bw_pop[i], clint_pop[i], fup_pop[i]
        Qh = 90*(bw/70)**0.75
        cl = clint*45*1500/1000
        CLh = (Qh*cl*fup)/(Qh+cl*fup)
        Vd  = max(0.5, (0.2+0.8*d["logp"])*bw)
        C0  = (dose_mg_kg*bw*1000/d["mw"])*0.9/(Vd*1000)
        Cmaxs.append(float(C0))
    Cmaxs = np.array(Cmaxs)
    return {
        "Cmax_mean_uM":    round(Cmaxs.mean(), 4),
        "Cmax_sd_uM":      round(Cmaxs.std(), 4),
        "Cmax_p5_uM":      round(np.percentile(Cmaxs, 5), 4),
        "Cmax_p95_uM":     round(np.percentile(Cmaxs, 95), 4),
        "n_high_exposure": int((Cmaxs > np.percentile(Cmaxs, 95)).sum()),
        "n_subjects":      n_subjects,
        "replaces": "Animal TK study + population variability study",
        "3Rs_saving": f"~{n_subjects//5} animals (equivalent TK information)",
    }

def tool_cytotox_correction(bioactivity_ac50_uM: float,
                              cytotox_ac50_uM: float) -> dict:
    """Flag HTS hits as cytotoxicity artefacts. Replaces follow-up assays."""    ratio = bioactivity_ac50_uM / cytotox_ac50_uM if cytotox_ac50_uM > 0 else 0
    return {
        "bioactivity_ac50_uM": bioactivity_ac50_uM,
        "cytotox_ac50_uM":     cytotox_ac50_uM,
        "burst_ratio":         round(ratio, 4),
        "call":     ("CYTOTOX_ARTEFACT" if ratio >= 0.1
                else "ACTIVE_SPECIFIC"),
        "flag":     ratio >= 0.1,
        "action":   ("DEPRIORITISE — likely non-specific" if ratio>=0.1
                else "PROCEED — activity below cytotox threshold"),
        "reference": "Judson 2016, EPA Tox21 burst ratio method",
    }

print("Tools 11-17 defined ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# TOOLS 18-25: PAINS, Genotox Battery, Reactive Met, Endocrine, AOP, Read-Across,
#              Literature Search, Regulatory Classification
# ══════════════════════════════════════════════════════════════════════════════

def tool_pains_filter(smiles: str) -> dict:
    """PAINS (pan-assay interference) filter. Removes HTS false positives."""    mol = _mol(smiles)
    params = FilterCatalogParams()
    params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
    catalog = FilterCatalog.FilterCatalog(params)
    entry = catalog.GetFirstMatch(mol)
    return {
        "pains_alert": bool(entry),
        "pattern": entry.GetDescription() if entry else None,
        "action": "REMOVE — PAINS compound causes HTS false positives" if entry else "CLEAN",
        "reference": "Baell & Holloway 2010 (J Med Chem)",
    }

def tool_genotox_battery(smiles: str) -> dict:
    """Full ICH S2(R1) in silico genotox battery. Replaces full 3-test animal battery."""    sa   = tool_structural_alerts(smiles)
    qsar = tool_qsar_ames(smiles)
    mol  = _mol(smiles)
    # In silico MNT (micronucleus test) proxy
    has_spindle = mol.HasSubstructMatch(Chem.MolFromSmarts("c1cccc2cccnc12")) or                   mol.HasSubstructMatch(Chem.MolFromSmarts("[#6]=[#6]CC"))
    # Chromosomal aberration proxy
    ca_concern  = mol.HasSubstructMatch(Chem.MolFromSmarts("[N+](=O)[O-]")) or                   mol.HasSubstructMatch(Chem.MolFromSmarts("[NH2]c1ccccc1"))
    overall = ("CONCERN" if (sa["sa_call"]=="POSITIVE" and qsar["qsar_call"]=="POSITIVE")
               or sa["ich_m7_class"].startswith("Class 1")
          else "INVESTIGATE" if sa["sa_call"]=="POSITIVE" or qsar["qsar_call"]=="POSITIVE"
          else "NO CONCERN")
    return {
        "ames_sa_call":    sa["sa_call"],
        "ames_qsar_call":  qsar["qsar_call"],
        "MNT_concern":     bool(has_spindle),
        "CA_concern":      bool(ca_concern),
        "overall":         overall,
        "ich_s2_r1":       overall,
        "replaces": "Ames test (bacteria) + MNT (mouse) + CA (CHO/human lymphocyte)",
        "guideline": "ICH S2(R1) 2012, OECD TG 471/487/473",
        "3Rs_saving": "~50 mice (MNT) + mammalian cell assay avoided per compound",
    }

def tool_reactive_metabolite(smiles: str) -> dict:
    """Reactive metabolite risk assessment. Replaces protein covalent binding study."""    mol = _mol(smiles); d = _desc(mol)
    alerts = {
        "para_aminophenol": mol.HasSubstructMatch(Chem.MolFromSmarts("Nc1ccc(O)cc1")),
        "aniline":          mol.HasSubstructMatch(Chem.MolFromSmarts("[NH2]c1ccccc1")),
        "furan":            mol.HasSubstructMatch(Chem.MolFromSmarts("c1ccoc1")),
        "thiophene":        mol.HasSubstructMatch(Chem.MolFromSmarts("c1ccsc1")),
        "acyl_glucuronide": d["mw"] > 300 and mol.HasSubstructMatch(Chem.MolFromSmarts("C(=O)O")),
        "michael_acceptor": mol.HasSubstructMatch(Chem.MolFromSmarts("C=CC=O")),
        "epoxide_forming":  mol.HasSubstructMatch(Chem.MolFromSmarts("c1ccc2c(c1)cc1ccccc1c2")),
    }
    n_alerts = sum(1 for v in alerts.values() if v)
    risk = "HIGH" if n_alerts >= 3 else "MODERATE" if n_alerts >= 1 else "LOW"
    return {
        "alerts_found":    {k: v for k, v in alerts.items() if v},
        "n_rm_alerts":     n_alerts,
        "rm_risk":         risk,
        "GSH_trapping_prediction": risk in ("HIGH","MODERATE"),
        "replaces": "GSH trapping / protein binding (in vitro, but eliminates animal follow-up)",
        "recommendation": f"{'Conduct GSH trapping assay in vitro' if risk!='LOW' else 'No further RM testing needed'}",
    }

def tool_endocrine_disruption(smiles: str) -> dict:
    """Endocrine disruption screening (ER/AR/AhR). Replaces in vivo uterotrophic."""    np.random.seed(SEED + 21)
    mol = _mol(smiles); d = _desc(mol)
    # Structural similarity to steroid scaffold (simplified)
    phenol    = mol.HasSubstructMatch(Chem.MolFromSmarts("Oc1ccccc1"))
    biphenol  = mol.HasSubstructMatch(Chem.MolFromSmarts("Oc1ccc(cc1)c1ccc(O)cc1"))
    dioxin_like=mol.HasSubstructMatch(Chem.MolFromSmarts("c1ccc2c(c1)Oc1ccccc1O2"))
    er_score  = float(0.4*biphenol + 0.25*phenol + np.random.normal(0,0.05))
    ar_score  = float(0.3*phenol   + 0.15*(d["logp"]>4) + np.random.normal(0,0.05))
    ahr_score = float(0.5*dioxin_like + 0.2*(d["naro"]>=3) + np.random.normal(0,0.05))
    return {
        "ER_agonism_score":   round(np.clip(er_score,0,1),3),
        "AR_agonism_score":   round(np.clip(ar_score,0,1),3),
        "AhR_activation":     round(np.clip(ahr_score,0,1),3),
        "ED_concern":         er_score>0.3 or ar_score>0.3 or ahr_score>0.4,
        "calls": {
            "ER": "CONCERN" if er_score>0.3 else "NO CONCERN",
            "AR": "CONCERN" if ar_score>0.3 else "NO CONCERN",
            "AhR":"CONCERN" if ahr_score>0.4 else "NO CONCERN",
        },
        "replaces": "Uterotrophic assay (rat), Hershberger assay (rat), H295R steroidogenesis",
        "guideline": "OECD TG 455 (ER), TG 457 (AR), EPA EDSP Tier 1",
    }

def tool_aop_scoring(smiles: str) -> dict:
    """Score compound against key AOPs. Replaces mechanistic animal studies."""    mol = _mol(smiles); d = _desc(mol)
    aop_scores = {}
    # AOP #54: NIS inhibition → thyroid
    aop_scores["AOP54_thyroid"] = float(np.clip(
        0.3*(d["mw"]>400) + 0.2*(d["logp"]>3) + np.random.normal(0,0.05), 0, 1))
    # AOP #8: Aromatase inhibition → reproductive
    aop_scores["AOP8_aromatase"]= float(np.clip(
        0.25*mol.HasSubstructMatch(Chem.MolFromSmarts("Oc1ccc(cc1)c1ccc(O)cc1"))
        + 0.15*(d["logp"]>2) + np.random.normal(0,0.05), 0, 1))
    # AOP #57: Mitochondrial complex I inhibition → liver
    aop_scores["AOP57_mito_cx1"]= float(np.clip(
        0.3*(d["mw"]>300 and d["logp"]>2) + np.random.normal(0,0.05), 0, 1))
    triggered = {k: v for k, v in aop_scores.items() if v > 0.3}
    return {
        "aop_scores": {k: round(v,3) for k,v in aop_scores.items()},
        "triggered_AOPs": list(triggered.keys()),
        "mechanistic_concerns": len(triggered) > 0,
        "replaces": "Mechanistic in vivo studies, mode of action investigations",
        "database": "OECD AOP-Wiki (https://aopwiki.org/)",
    }

_TOX_LITERATURE = {
    "ICH M7(R2)":    "Two complementary in silico methods required for impurity genotox. Sensitivity >=90%.",
    "OECD TG 497":   "Defined Approach 2o3 (DPRA+KS+hCLAT) validated 2023. No LLNA needed.",
    "ICH E14/S7B":   "CiPA multi-channel replaces hERG-only. hiPS-CM MEA = Tier 2 (2022).",
    "DILIrank":      "1036 FDA drugs classified. iPSC spheroids 85%+ concordance with in vivo.",
    "EPA HTTK":      "IVIVE via 3-compartment PBPK. CLint+fup input. AED output.",
    "FDA FMA 2.0":   "FDA Modernization Act 2.0 (2022): animal data no longer required for IND.",
    "OECD GD 255":   "IATA framework for structured weight-of-evidence without animal data.",
    "CiPA":          "Comprehensive In vitro Proarrhythmia Assay — FDA/HESI/EMA initiative 2016-2022.",
}

def tool_literature_search(query: str) -> dict:
    """Search curated regulatory toxicology knowledge base for evidence."""    query_lower = query.lower()
    hits = {}
    for key, text in _TOX_LITERATURE.items():
        if any(w in query_lower for w in key.lower().split() + text.lower().split()[:5]):
            hits[key] = text
    if not hits:
        hits = {list(_TOX_LITERATURE.keys())[0]: list(_TOX_LITERATURE.values())[0]}
    return {"query": query, "n_hits": len(hits), "results": hits,
            "note": "Simulated KB — connect to PubMed/ChEMBL API for production"}

def tool_regulatory_classification(results_dict: dict) -> dict:
    """
    Final OECD GD 255 IATA weight-of-evidence classification.
    Integrates all NAM results into a regulatory safety conclusion.
    """
    concerns, positives = [], []
    if results_dict.get("genotox_overall") in ("CONCERN","POSITIVE"):
        concerns.append("Genotoxicity concern (ICH M7)")
    if results_dict.get("herg_risk") == "HIGH":
        concerns.append("hERG concern (ICH S7B)")
    if results_dict.get("cipa_tdp") == "HIGH TdP":
        concerns.append("CiPA TdP risk")
    if results_dict.get("dili_class") == "Most-concern":
        concerns.append("DILI Most-concern (DILIrank)")
    if results_dict.get("skin_hazard") == "SENSITISER":
        concerns.append("Skin sensitisation hazard (OECD TG 497)")
    if results_dict.get("ed_concern"):
        concerns.append("Endocrine disruption concern")
    if results_dict.get("rm_risk") == "HIGH":
        concerns.append("Reactive metabolite risk")
    n_clear = sum([
        results_dict.get("genotox_overall") == "NO CONCERN",
        results_dict.get("herg_risk") == "LOW",
        results_dict.get("dili_class") == "No-concern",
        results_dict.get("skin_hazard") == "NON-SENSITISER",
    ])
    if not concerns and n_clear >= 3:
        conclusion = "NO SIGNIFICANT HAZARD IDENTIFIED"
        confidence = "HIGH" if n_clear >= 4 else "MODERATE"
        next_step  = "Proceed to Phase I clinical study (if applicable)"
    elif len(concerns) <= 1:
        conclusion = "LIMITED CONCERN — MONITOR"
        confidence = "MODERATE"
        next_step  = "Address flagged concern with targeted in vitro assay"
    else:
        conclusion = f"HAZARD IDENTIFIED ({len(concerns)} concerns)"
        confidence = "HIGH"
        next_step  = "Structural modification required before progression"
    return {
        "conclusion":      conclusion,
        "confidence":      confidence,
        "n_concerns":      len(concerns),
        "concerns":        concerns,
        "n_clear_endpoints":n_clear,
        "next_step":       next_step,
        "framework":       "OECD GD 255 IATA / FDA Modernization Act 2.0",
        "animal_studies_required": len(concerns) > 2,
        "animal_tests_avoided": "Full battery (~200 animals) if no concerns identified",
    }

print("Tools 18-25 defined ✓")

# ── Assemble master tool registry ─────────────────────────────────────────────
TOOLS = {
    "admet":              ToxTool("admet","Full ADMET profile: MW, LogP, TPSA, Ro5, QED, BBB, CNS MPO","ADMET","Physico-chem screening","Ro5/Veber", tool_admet_profile),
    "structural_alerts":  ToxTool("structural_alerts","ICH M7 structural alert SMARTS (21 patterns, Classes 1-3)","Genotoxicity","Ames screen","ICH M7(R2)", tool_structural_alerts),
    "qsar_ames":          ToxTool("qsar_ames","QSAR Ames mutagenicity (RF-ECFP4). ICH M7 Method 2","Genotoxicity","Ames test","ICH M7(R2)", tool_qsar_ames),
    "qsar_ld50":          ToxTool("qsar_ld50","Predict acute oral LD50 and GHS category","Acute toxicity","OECD 423 LD50","OECD 423", tool_qsar_ld50),
    "herg":               ToxTool("herg","hERG IC50 prediction for cardiac safety screening","Cardiac","hERG patch-clamp","ICH S7B", tool_herg_ic50),
    "cipa":               ToxTool("cipa","CiPA 7-channel ion current model. TdP risk score","Cardiac","hERG-only ICH S7B","ICH E14/S7B", tool_cipa_multichannel),
    "hipscm_mea":         ToxTool("hipscm_mea","hiPS-CM MEA electrophysiology. CiPA Tier 2. FPDc, EAD","Cardiac","In vivo QT study","CiPA Tier 2", tool_hipscm_mea),
    "dili":               ToxTool("dili","DILI severity prediction (DILIrank ML). Hepatotox mechanisms","Hepatotoxicity","Rat liver study","DILIrank", tool_dili_prediction),
    "liver_organoid":     ToxTool("liver_organoid","3D liver organoid (iPSC spheroid). 14-day DILI assay","Hepatotoxicity","28-day rat liver","InSphero/CN Bio", tool_liver_organoid),
    "ooc_liver":          ToxTool("ooc_liver","Liver organ-on-chip TEER/albumin/CYP/urea/bile","Hepatotoxicity","Repeat-dose hepatotox","OECD TG 407", tool_ooc_liver),
    "ooc_kidney":         ToxTool("ooc_kidney","Kidney OoC KIM-1/NGAL/creatinine clearance","Nephrotoxicity","Rat nephrotox","OECD TG 407", tool_ooc_kidney),
    "skin_sens":          ToxTool("skin_sens","OECD TG 497 DA2 (2o3): DPRA+KeratinoSens+hCLAT","Skin sensitisation","LLNA mouse","OECD TG 497", tool_skin_sensitisation),
    "bbb":                ToxTool("bbb","BBB penetration score + CNS MPO + Kp_uu estimate","CNS","Brain distribution study","CNS MPO", tool_bbb_penetration),
    "neurotox":           ToxTool("neurotox","DNT Tier 1: AChE, NMDA, BBB, excitotoxicity screen","Neurotoxicity","Rat DNT study","ICH S5(R3)", tool_neurotox_screen),
    "pbpk":               ToxTool("pbpk","EPA HTTK 3-compartment IVIVE. CLint+fup → AED → TTC","TK/IVIVE","Dose range-finding","EPA HTTK", tool_pbpk_ivive),
    "pop_pbpk":           ToxTool("pop_pbpk","Monte Carlo population PBPK (n=200). Cmax distribution","TK population","Animal TK study","Jamei 2009", tool_population_pbpk),
    "cytotox_corr":       ToxTool("cytotox_corr","Cytotoxicity burst ratio correction for HTS hits","HTS QC","Follow-up assays","Judson 2016", tool_cytotox_correction),
    "pains":              ToxTool("pains","PAINS filter (pan-assay interference removal)","HTS QC","False-positive assays","Baell 2010", tool_pains_filter),
    "genotox_battery":    ToxTool("genotox_battery","Full ICH S2(R1) in silico battery: Ames+MNT+CA","Genotoxicity","Ames+MNT+CA animal tests","ICH S2(R1)", tool_genotox_battery),
    "reactive_met":       ToxTool("reactive_met","Reactive metabolite structural alert screen","Bioactivation","GSH trapping/protein binding","Kalgutkar 2005", tool_reactive_metabolite),
    "endocrine":          ToxTool("endocrine","ED screen: ER/AR/AhR. Replaces in vivo assays","Endocrine","Uterotrophic/Hershberger","OECD TG 455/457", tool_endocrine_disruption),
    "aop":                ToxTool("aop","Score compound vs OECD AOP-Wiki key events","Mechanism","Mechanistic animal studies","OECD AOP-Wiki", tool_aop_scoring),
    "read_across":        ToxTool("read_across","Tanimoto-based category formation for data-gap filling","Data gaps","New animal tests","OECD RAAF", None),
    "literature":         ToxTool("literature","Search regulatory tox KB: ICH, OECD, FDA, EPA","Literature","Expert review","OECD GD 255", tool_literature_search),
    "regulatory":         ToxTool("regulatory","OECD GD 255 IATA WoE final classification","WoE conclusion","Full animal battery","OECD GD 255", tool_regulatory_classification),
}

# ── Fix read_across (requires special signature) ──────────────────────────────
def _read_across_fn(smiles: str, library_smiles: list = None) -> dict:
    mol = _mol(smiles)
    if library_smiles is None:
        library_smiles = ["CC(=O)Oc1ccccc1C(=O)O","Cn1cnc2c1c(=O)n(C)c(=O)n2C",
                          "CC(C)Cc1ccc(cc1)C(C)C(=O)O"]
    tgt_fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048)
    analogs = []
    for s in library_smiles:
        m = Chem.MolFromSmiles(s)
        if m:
            fp = AllChem.GetMorganFingerprintAsBitVect(m,2,2048)
            tc = DataStructs.TanimotoSimilarity(tgt_fp, fp)
            if tc > 0.3:
                analogs.append({"smiles": s, "tanimoto": round(tc,4)})
    analogs.sort(key=lambda x: -x["tanimoto"])
    return {"target": smiles, "analogs_found": len(analogs),
            "top_analogs": analogs[:5],
            "in_AD": len(analogs) > 0,
            "replaces": "New animal test for data gap filling",
            "guideline": "OECD RAAF 2017"}
TOOLS["read_across"].func = _read_across_fn

TOOL_DESCRIPTIONS = "\n".join(f"  {k}: {t.description}" for k, t in TOOLS.items())
print(f"Tool registry: {len(TOOLS)} tools registered ✓")

---
## Section 3 — Eight Specialist Agents

Each specialist agent has domain expertise and a curated subset of tools.

In [ ]:
# ── Specialist agent definitions ─────────────────────────────────────────────

@dataclass
class SpecialistAgent:
    name:        str
    role:        str
    tools:       list[str]         # tool names this agent uses
    keywords:    list[str]         # routing keywords
    runs:        int = 0
    _results:    dict = field(default_factory=dict)

    def run(self, smiles: str, context: dict = None) -> dict:
        self.runs += 1
        results  = {}
        for tool_name in self.tools:
            tool = TOOLS.get(tool_name)
            if tool is None or tool.func is None:
                continue
            try:
                raw = tool.func(smiles=smiles)
                results[tool_name] = raw if isinstance(raw, dict) else json.loads(raw)
            except Exception as e:
                results[tool_name] = {"error": str(e)}
        self._results = results
        return results

# Define the eight specialists
AGENTS = {

"admet_agent": SpecialistAgent(
    name="ADMET Pharmacologist",
    role="Physicochemical profiling, bioavailability, distribution, metabolism, excretion",
    tools=["admet", "bbb", "pbpk", "pop_pbpk"],
    keywords=["admet","bioavail","logp","mw","bbb","cns","tk","distribution","absorption",
               "clearance","half-life","pbpk","ivive"],
),

"geno_agent": SpecialistAgent(
    name="Genetic Toxicologist",
    role="Genotoxicity and mutagenicity assessment per ICH M7 and ICH S2(R1)",
    tools=["structural_alerts","qsar_ames","genotox_battery","pains"],
    keywords=["geno","mutagen","ames","dna","clastogen","chromosome","mntest","ich m7",
               "ich s2","structural alert","nitrosamine","nitroso"],
),

"cardiac_agent": SpecialistAgent(
    name="Cardiac Safety Scientist",
    role="Cardiac electrophysiology and arrhythmia risk per CiPA/ICH E14/S7B",
    tools=["herg","cipa","hipscm_mea"],
    keywords=["cardiac","herg","qt","fpd","arrhythm","cipa","tdp","torsades","ion channel",
               "ecg","qrs","ich s7b","ich e14","ead"],
),

"dili_agent": SpecialistAgent(
    name="Hepatotoxicologist",
    role="Drug-induced liver injury prediction using organoids and OoC systems",
    tools=["dili","liver_organoid","ooc_liver","reactive_met"],
    keywords=["liver","hepato","dili","alt","ast","albumin","cyp","hepatocyte","organoid",
               "steatosis","cholestasis","biliary","bsep","reactive metabolite"],
),

"organ_agent": SpecialistAgent(
    name="Organ Toxicologist (Non-Hepatic)",
    role="Multi-organ toxicity: kidney, CNS, skin sensitisation using OoC/organoids",
    tools=["ooc_kidney","neurotox","skin_sens","endocrine","aop"],
    keywords=["kidney","renal","nephro","neuro","brain","dnt","skin","sensitise","endocrine",
               "ed","thyroid","reproductive","aop","adverse outcome","kim1","ngal"],
),

"tk_agent": SpecialistAgent(
    name="Toxicokinetics / IVIVE Specialist",
    role="Dosimetry bridge between in vitro and in vivo: PBPK, IVIVE, TTC",
    tools=["pbpk","pop_pbpk","cytotox_corr"],
    keywords=["pbpk","ivive","ttc","aed","dose","css","cmax","clearance","bioavail",
               "extrapolation","population","variability","cytotox","ac50","ec50"],
),

"mechanism_agent": SpecialistAgent(
    name="Mechanistic Toxicologist",
    role="Mode of action, AOP analysis, reactive metabolite, endocrine disruption",
    tools=["reactive_met","endocrine","aop","read_across","qsar_ld50"],
    keywords=["mechanism","mode of action","aop","pathway","reactive","gsh","gssh",
               "endocrine","er","ar","ahr","read-across","category","analogue","ld50"],
),

"regulatory_agent": SpecialistAgent(
    name="Regulatory Affairs Scientist",
    role="IATA WoE integration, literature synthesis, regulatory classification",
    tools=["literature","regulatory"],
    keywords=["regulatory","iata","woe","weight of evidence","oecd","ich","epa","fda",
               "submission","dossier","classification","conclusion","guideline","echa"],
),
}

print(f"Specialist agents: {len(AGENTS)} defined")
for name, agent in AGENTS.items():
    print(f"  {agent.name:40s} → {len(agent.tools)} tools")

---
## Section 4 — LangGraph State Machine

The pipeline is a directed graph. Each node is a function. Edges are conditional. The `ToxState` TypedDict tracks everything across nodes.

In [ ]:
# ── TypedDict state ───────────────────────────────────────────────────────────
from typing import TypedDict

class ToxState(TypedDict):
    # Input
    smiles:          str
    compound_name:   str
    intended_use:    str          # pharmaceutical / pesticide / industrial

    # Pipeline control
    step:            str
    steps_completed: list[str]
    errors:          list[str]

    # Agent results (one dict per specialist)
    admet_results:    dict
    geno_results:     dict
    cardiac_results:  dict
    dili_results:     dict
    organ_results:    dict
    tk_results:       dict
    mechanism_results:dict
    regulatory_results:dict

    # Extracted key flags (set by router, used in edges)
    herg_risk:       str          # LOW / MEDIUM / HIGH
    dili_concern:    str          # NONE / MODERATE / HIGH
    geno_concern:    str          # NO CONCERN / INVESTIGATE / CONCERN
    skin_hazard:     str
    cipa_tdp:        str
    ed_concern:      bool
    rm_risk:         str

    # Uncertainty & confidence
    uncertain_endpoints: list[str]
    requires_hitl:   bool
    hitl_decision:   Optional[str]    # APPROVE / REJECT / MORE_DATA

    # Memory
    episodic_memory: list[dict]
    semantic_memory: dict

    # Final output
    final_report:    str
    dossier:         dict

def make_initial_state(smiles: str,
                        compound_name: str = "Unknown",
                        intended_use:  str = "pharmaceutical") -> ToxState:
    return ToxState(
        smiles=smiles, compound_name=compound_name, intended_use=intended_use,
        step="init", steps_completed=[], errors=[],
        admet_results={}, geno_results={}, cardiac_results={},
        dili_results={}, organ_results={}, tk_results={},
        mechanism_results={}, regulatory_results={},
        herg_risk="UNKNOWN", dili_concern="UNKNOWN", geno_concern="UNKNOWN",
        skin_hazard="UNKNOWN", cipa_tdp="UNKNOWN", ed_concern=False, rm_risk="LOW",
        uncertain_endpoints=[], requires_hitl=False, hitl_decision=None,
        episodic_memory=[], semantic_memory={},
        final_report="", dossier={},
    )

print("ToxState TypedDict defined ✓")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# PIPELINE NODES (each node = one graph step)
# ══════════════════════════════════════════════════════════════════════════════

def node_validate(state: ToxState) -> ToxState:
    """Node 1: Validate SMILES and standardise."""    mol = Chem.MolFromSmiles(state["smiles"])
    if mol is None:
        state["errors"].append("Invalid SMILES — cannot proceed")
        state["step"] = "error"
        return state
    # Standardise
    try:
        mol_std = rdMolStandardize.LargestFragmentChooser().choose(mol)
        mol_std = rdMolStandardize.Uncharger().uncharge(mol_std)
        state["smiles"] = Chem.MolToSmiles(mol_std)
    except Exception:
        pass  # keep original if standardisation fails
    state["step"] = "validated"
    state["steps_completed"].append("validate")
    return state

def node_admet(state: ToxState) -> ToxState:
    """Node 2: ADMET + TK profiling."""    state["admet_results"] = AGENTS["admet_agent"].run(state["smiles"])
    state["step"] = "admet_done"
    state["steps_completed"].append("admet")
    return state

def node_genotox(state: ToxState) -> ToxState:
    """Node 3: Genotoxicity (ICH M7 + ICH S2)."""    state["geno_results"] = AGENTS["geno_agent"].run(state["smiles"])
    # Extract key flag
    gb = state["geno_results"].get("genotox_battery", {})
    state["geno_concern"] = gb.get("overall", "UNKNOWN")
    state["step"] = "geno_done"
    state["steps_completed"].append("genotox")
    return state

def node_cardiac(state: ToxState) -> ToxState:
    """Node 4: Cardiac safety (CiPA + hiPS-CM MEA)."""    state["cardiac_results"] = AGENTS["cardiac_agent"].run(state["smiles"])
    herg = state["cardiac_results"].get("herg", {})
    cipa = state["cardiac_results"].get("cipa", {})
    mea  = state["cardiac_results"].get("hipscm_mea", {})
    state["herg_risk"] = herg.get("risk_level", "UNKNOWN")
    state["cipa_tdp"]  = cipa.get("TdP_category", "UNKNOWN")
    state["step"] = "cardiac_done"
    state["steps_completed"].append("cardiac")
    return state

def node_dili(state: ToxState) -> ToxState:
    """Node 5: DILI + liver organoid + OoC liver."""    state["dili_results"] = AGENTS["dili_agent"].run(state["smiles"])
    dili = state["dili_results"].get("dili", {})
    org  = state["dili_results"].get("liver_organoid", {})
    state["dili_concern"] = dili.get("DILIrank_class", "Unknown")
    state["rm_risk"]      = state["dili_results"].get("reactive_met",{}).get("rm_risk","LOW")
    state["step"] = "dili_done"
    state["steps_completed"].append("dili")
    return state

def node_organ(state: ToxState) -> ToxState:
    """Node 6: Multi-organ toxicity (kidney, CNS, skin, endocrine, AOP)."""    state["organ_results"] = AGENTS["organ_agent"].run(state["smiles"])
    skin = state["organ_results"].get("skin_sens", {})
    ed   = state["organ_results"].get("endocrine", {})
    state["skin_hazard"] = skin.get("hazard_call", "UNKNOWN")
    state["ed_concern"]  = bool(ed.get("ED_concern", False))
    state["step"] = "organ_done"
    state["steps_completed"].append("organ")
    return state

def node_mechanism(state: ToxState) -> ToxState:
    """Node 7: Mechanism, AOP, read-across, acute tox."""    state["mechanism_results"] = AGENTS["mechanism_agent"].run(state["smiles"])
    state["step"] = "mechanism_done"
    state["steps_completed"].append("mechanism")
    return state

def node_risk_tier(state: ToxState) -> ToxState:
    """Node 8: Compute overall risk tier and uncertainty."""    concerns = []
    if state["geno_concern"]  == "CONCERN":       concerns.append("Genotoxicity")
    if state["herg_risk"]     == "HIGH":           concerns.append("hERG")
    if state["cipa_tdp"]      == "HIGH TdP":       concerns.append("CiPA TdP")
    if state["dili_concern"]  == "Most-concern":   concerns.append("DILI")
    if state["skin_hazard"]   == "SENSITISER":     concerns.append("Skin sensitisation")
    if state["ed_concern"]:                        concerns.append("Endocrine disruption")
    if state["rm_risk"]       == "HIGH":           concerns.append("Reactive metabolite")

    # Uncertain if outside AD or conflicting signals
    uncertain = []
    if state["geno_concern"]   == "INVESTIGATE":  uncertain.append("Genotoxicity")
    if state["herg_risk"]      == "MEDIUM":        uncertain.append("Cardiac")
    if state["dili_concern"]   == "Less-concern":  uncertain.append("Hepatotoxicity")
    state["uncertain_endpoints"] = uncertain

    # HITL required for HIGH risk or multiple conflicting results
    state["requires_hitl"] = len(concerns) >= 2 or (
        "Genotoxicity" in concerns and len(concerns) >= 1)

    state["semantic_memory"]["active_concerns"] = concerns
    state["semantic_memory"]["n_concerns"]      = len(concerns)
    state["step"] = "risk_tiered"
    state["steps_completed"].append("risk_tier")
    return state

def node_hitl(state: ToxState) -> ToxState:
    """Node 9: Human-in-the-loop review for HIGH risk compounds."""    concerns = state["semantic_memory"].get("active_concerns", [])
    print(f"\n  ╔{'═'*60}╗")
    print(f"  ║  HUMAN-IN-THE-LOOP REVIEW REQUIRED                     ║")
    print(f"  ║  Compound: {state['compound_name']:20s}                      ║")
    print(f"  ║  Concerns: {', '.join(concerns)[:45]:45s}  ║")
    print(f"  ╠{'═'*60}╣")
    print(f"  ║  Auto-decision: REFER FOR EXPERT REVIEW                ║")
    print(f"  ║  (In production: block pipeline, notify toxicologist)   ║")
    print(f"  ╚{'═'*60}╝")
    # Auto-decision for tutorial; in production: await human input
    n = state["semantic_memory"].get("n_concerns", 0)
    state["hitl_decision"] = "REJECT" if n >= 3 else "MORE_DATA" if n >= 1 else "APPROVE"
    state["steps_completed"].append("hitl")
    state["step"] = "hitl_done"
    return state

def node_regulatory(state: ToxState) -> ToxState:
    """Node 10: IATA WoE regulatory classification."""    consolidated = {
        "genotox_overall": state["geno_concern"],
        "herg_risk":       state["herg_risk"],
        "cipa_tdp":        state["cipa_tdp"],
        "dili_class":      state["dili_concern"],
        "skin_hazard":     state["skin_hazard"],
        "ed_concern":      state["ed_concern"],
        "rm_risk":         state["rm_risk"],
    }
    state["regulatory_results"] = tool_regulatory_classification(consolidated)
    state["steps_completed"].append("regulatory")
    state["step"] = "regulatory_done"
    return state


def node_report(state: ToxState) -> ToxState:
    """Node 11: Generate final regulatory dossier."""    reg    = state["regulatory_results"]
    admet  = state["admet_results"].get("admet", {})
    herg   = state["cardiac_results"].get("herg", {})
    mea    = state["cardiac_results"].get("hipscm_mea", {})
    org    = state["dili_results"].get("liver_organoid", {})
    skin   = state["organ_results"].get("skin_sens", {})
    pbpk   = state["admet_results"].get("pbpk", {})
    geno   = state["geno_results"].get("genotox_battery", {})

    animals_avoided = sum([
        state["geno_concern"]   != "CONCERN",      # Ames + MNT + CA battery
        state["herg_risk"]       != "HIGH",          # hERG + in vivo QT
        state["dili_concern"]    != "Most-concern",  # 28-day hepatotox
        state["skin_hazard"]     == "NON-SENSITISER",# LLNA
        not state["ed_concern"],                     # uterotrophic/Hershberger
    ])
    est_animals = animals_avoided * 12 + (15 if state["geno_concern"]!="CONCERN" else 0)

    state["final_report"] = _build_report(
        state, admet, herg, mea, org, skin, pbpk, geno, reg, est_animals)
        state["dossier"] = {
        "compound_name":  state["compound_name"],
        "smiles":         state["smiles"],
        "conclusion":     reg.get("conclusion"),
        "confidence":     reg.get("confidence"),
        "concerns":       reg.get("concerns",[]),
        "animals_avoided":est_animals,
        "hitl_decision":  state.get("hitl_decision"),
        "steps":          state["steps_completed"],
    }
    state["step"] = "done"
    return state

# ── Conditional edges ─────────────────────────────────────────────────────────
def route_after_risk(state: ToxState) -> Literal["hitl","regulatory"]:
    return "hitl" if state["requires_hitl"] else "regulatory"

print("All pipeline nodes defined ✓")

---
## Section 5 — Supervisor Agent & Full Pipeline Execution

In [ ]:
# ── Supervisor: routes queries to specialist agents ───────────────────────────

class SupervisorAgent:
    """
    Coordinates all specialist agents, manages the pipeline state,
    and assembles the final IATA WoE dossier.
    """
    def __init__(self):
        self.run_count   = 0
        self.memory      = deque(maxlen=20)

    def run_full_pipeline(self, smiles: str,
                           compound_name: str = "Unknown",
                           intended_use: str  = "pharmaceutical",
                           verbose: bool      = True) -> ToxState:
        """
        Execute the complete animal-free toxicology pipeline:
          validate → admet → genotox → cardiac → dili
          → organ → mechanism → risk_tier → [hitl] → regulatory → report
        """
        self.run_count += 1
        state = make_initial_state(smiles, compound_name, intended_use)

        nodes = [
            ("Validating SMILES...",              node_validate),
            ("Running ADMET + TK profiling...",   node_admet),
            ("Running genotoxicity battery...",   node_genotox),
            ("Running cardiac safety (CiPA)...",  node_cardiac),
            ("Running DILI + liver organoid...",  node_dili),
            ("Running multi-organ OoC...",        node_organ),
            ("Running mechanistic analysis...",   node_mechanism),
            ("Computing risk tier...",            node_risk_tier),
        ]

        for msg, node_fn in nodes:
            if state["step"] == "error":
                break
            if verbose:
                print(f"  [{state['compound_name'][:20]:20s}] {msg}")
            state = node_fn(state)

        # Conditional routing
        next_node = route_after_risk(state)
        if verbose:
            print(f"  [{'Routing':20s}] → {next_node}")
        if next_node == "hitl":
            state = node_hitl(state)

        state = node_regulatory(state)
        state = node_report(state)

        # Save to episodic memory
        self.memory.append({
            "smiles":     smiles[:30],
            "name":       compound_name,
            "conclusion": state["dossier"].get("conclusion","N/A"),
            "concerns":   len(state["dossier"].get("concerns",[])),
        })
        return state

supervisor = SupervisorAgent()
print("SupervisorAgent ready ✓")

In [ ]:
# ── Run the full pipeline on a panel of compounds ─────────────────────────────
COMPOUND_PANEL = [
    # (name, smiles, intended_use)
    ("Aspirin",
     "CC(=O)Oc1ccccc1C(=O)O",
     "pharmaceutical"),

    ("Troglitazone (hepatotoxic)",
     "Cc1ccc(CC2SC(=O)NC2=O)cc1OCC(C)(C)c1ccc(O)cc1",
     "pharmaceutical"),

    ("Nitrobenzene (genotoxic flag)",
     "O=[N+]([O-])c1ccccc1",
     "industrial"),

    ("Bisphenol A (endocrine disruptor)",
     "CC(C)(c1ccc(O)cc1)c1ccc(O)cc1",
     "industrial"),

    ("Cisapride (hERG risk)",
     "COCCNC(=O)c1cc(Cl)c(N)cc1OC1CCNCC1",
     "pharmaceutical"),
]

results_panel = {}
print("Running ToxAgent Pro — Full Panel")
print("=" * 65)
for name, smiles, use in COMPOUND_PANEL:
    print(f"\nCompound: {name}")
    state = supervisor.run_full_pipeline(smiles, name, use, verbose=True)
    results_panel[name] = state
    dos = state["dossier"]
    print(f"  → {dos['conclusion']}  [{dos['confidence']} confidence]")
    if dos["concerns"]:
        print(f"     Concerns: {', '.join(dos['concerns'])}")
    print(f"     ~{dos['animals_avoided']} animals avoided")

---
## Section 6 — Memory, Uncertainty & HITL

In [ ]:
# ── 6.1 Memory system ─────────────────────────────────────────────────────────

class ToxAgentMemory:
    """
    Dual memory system:
      Episodic  — recent compound assessments (bounded deque)
      Semantic  — distilled toxicology facts, calibration data, rules
    """
    def __init__(self, max_episodes: int = 50):
        self.episodic  = deque(maxlen=max_episodes)
        self.semantic  = {
            "false_positive_rate": 0.08,   # from validation study
            "sensitivity_ames":    0.92,    # model sensitivity
            "ad_threshold":        0.40,    # Tanimoto AD cutoff
            "confirmed_actives":   set(),
            "confirmed_inactives": set(),
            "structural_rules":    [
                "Nitrosamines are always ICH M7 Class 1 — no exceptions",
                "Ro5 violators with >2 violations rarely achieve oral bioavailability",
                "hERG IC50 < 1 µM requires full CiPA multi-channel",
                "DILIrank Most-concern requires organoid validation",
            ]
        }

    def store_result(self, state: ToxState):
        dos = state.get("dossier", {})
        self.episodic.append({
            "smiles":     state["smiles"],
            "name":       state["compound_name"],
            "conclusion": dos.get("conclusion","N/A"),
            "concerns":   dos.get("concerns",[]),
            "timestamp":  time.time(),
        })
        if dos.get("conclusion","").startswith("NO SIGNIFICANT"):
            self.semantic["confirmed_inactives"].add(state["smiles"])
        elif "HAZARD" in dos.get("conclusion",""):
            self.semantic["confirmed_actives"].add(state["smiles"])

    def recall_similar(self, smiles: str, n: int = 3) -> list[dict]:
        """Find similar past assessments by Tanimoto."""        mol = Chem.MolFromSmiles(smiles)
        if mol is None: return []
        tgt_fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, 2048)
        ranked = []
        for ep in self.episodic:
            m = Chem.MolFromSmiles(ep["smiles"]) if ep.get("smiles") else None
            if m:
                fp = AllChem.GetMorganFingerprintAsBitVect(m, 2, 2048)
                tc = DataStructs.TanimotoSimilarity(tgt_fp, fp)
                ranked.append((tc, ep))
        ranked.sort(key=lambda x: -x[0])
        return [{"tanimoto": round(tc,3), **ep} for tc, ep in ranked[:n]]

    def get_rules(self) -> list[str]:
        return self.semantic["structural_rules"]

# Populate memory with panel results
memory = ToxAgentMemory()
for name, state in results_panel.items():
    memory.store_result(state)

# Test recall
test_smi = "O=C(O)Cc1ccccc1Nc1c(Cl)cccc1Cl"  # diclofenac
similar  = memory.recall_similar(test_smi, n=3)
print("Episodic memory recall for Diclofenac-like compound:")
for ep in similar:
    print(f"  Tc={ep['tanimoto']:.3f}  {ep['name']:25s} → {ep['conclusion'][:40]}")

print("\nSemantic rules stored:")
for rule in memory.get_rules():
    print(f"  • {rule}")

In [ ]:
# ── 6.2 Conformal uncertainty quantification ──────────────────────────────────
def compute_uncertainty_profile(state: ToxState) -> dict:
    """
    Compute uncertainty for each endpoint.
    Flags compounds for additional assay vs regulatory confidence.
    """
    uncertain = []
    confident = []

    # Genotoxicity uncertainty
    geno = state["geno_results"]
    sa_call   = geno.get("structural_alerts",{}).get("sa_call","UNKNOWN")
    qsar_call = geno.get("qsar_ames",{}).get("qsar_call","UNKNOWN")
    in_ad     = geno.get("qsar_ames",{}).get("within_AD", True)

    if sa_call != qsar_call:
        uncertain.append(("Genotoxicity", "DISCORDANT — SA and QSAR disagree"))
    elif not in_ad:
        uncertain.append(("Genotoxicity", "QSAR outside applicability domain"))
    else:
        confident.append("Genotoxicity")

    # Cardiac uncertainty
    herg_r = state["herg_risk"]
    mea_r  = state["cardiac_results"].get("hipscm_mea",{}).get("risk_tier","UNKNOWN")
    if herg_r == "MEDIUM" or mea_r == "MODERATE":
        uncertain.append(("Cardiac", f"hERG={herg_r}, MEA={mea_r} — intermediate zone"))
    elif herg_r != mea_r.replace("MODERATE","MEDIUM"):
        uncertain.append(("Cardiac", "hERG and MEA not fully concordant"))
    else:
        confident.append("Cardiac")

    # DILI uncertainty
    dili_r = state["dili_concern"]
    org_r  = state["dili_results"].get("liver_organoid",{}).get("DILI_risk","UNKNOWN")
    if dili_r != org_r.replace("HIGH","Most-concern").replace("NONE","No-concern"):
        uncertain.append(("Hepatotoxicity", f"DILIrank={dili_r}, Organoid={org_r}"))
    else:
        confident.append("Hepatotoxicity")

    overall_confidence = (
        "HIGH"     if len(uncertain) == 0
        else "MODERATE" if len(uncertain) <= 2
        else "LOW"
    )

    return {
        "uncertain_endpoints":  uncertain,
        "confident_endpoints":  confident,
        "overall_confidence":   overall_confidence,
        "n_uncertain":          len(uncertain),
        "recommendation": (
            "Proceed with regulatory submission" if len(uncertain)==0
            else f"Run targeted follow-up for: {', '.join(e[0] for e in uncertain)}"
        ),
    }

# Compute for each compound in panel
print("Uncertainty Profiles:")
print(f"{'Compound':25s} {'Confidence':12s} {'Uncertain endpoints'}")
print("-"*70)
for name, state in results_panel.items():
    uq = compute_uncertainty_profile(state)
    unc_str = ", ".join(e[0] for e in uq["uncertain_endpoints"]) or "None"
    print(f"{name:25s} {uq['overall_confidence']:12s} {unc_str}")

---
## Section 7 — Visualisation Dashboard

In [ ]:
# ── 7.1 Comprehensive dashboard ───────────────────────────────────────────────
def plot_toxagent_dashboard(results_panel: dict, figsize=(20, 16)):
    """Generate a complete visual toxicology dashboard for the compound panel."""
    n_cpds   = len(results_panel)
    names    = list(results_panel.keys())
    colours  = ["#E74C3C","#E67E22","#27AE60","#1565C0","#8E44AD"]

    fig = plt.figure(figsize=figsize)
    gs  = gridspec.GridSpec(3, 3, figure=fig, hspace=0.50, wspace=0.40)

    # ── 1. Risk tier heatmap ──────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, :2])
    endpoints_list = ["Geno", "hERG", "CiPA TdP", "DILI", "Skin", "Endocrine", "RM Risk"]
    risk_matrix    = np.zeros((n_cpds, len(endpoints_list)))
    for i, (name, state) in enumerate(results_panel.items()):
        risk_matrix[i, 0] = 2 if state["geno_concern"]=="CONCERN"    else 1 if state["geno_concern"]=="INVESTIGATE" else 0
        risk_matrix[i, 1] = 2 if state["herg_risk"]=="HIGH"           else 1 if state["herg_risk"]=="MEDIUM"         else 0
        risk_matrix[i, 2] = 2 if "HIGH" in state["cipa_tdp"]          else 1 if "INTER" in state["cipa_tdp"]         else 0
        risk_matrix[i, 3] = 2 if state["dili_concern"]=="Most-concern"else 1 if state["dili_concern"]=="Less-concern" else 0
        risk_matrix[i, 4] = 2 if state["skin_hazard"]=="SENSITISER"   else 0
        risk_matrix[i, 5] = 2 if state["ed_concern"]                  else 0
        risk_matrix[i, 6] = 2 if state["rm_risk"]=="HIGH"             else 1 if state["rm_risk"]=="MODERATE" else 0

    from matplotlib.colors import ListedColormap
    cmap = ListedColormap(["#2ECC71","#F39C12","#E74C3C"])
    im = ax1.imshow(risk_matrix, cmap=cmap, vmin=0, vmax=2, aspect="auto")
    ax1.set_xticks(range(len(endpoints_list))); ax1.set_xticklabels(endpoints_list, fontsize=10)
    ax1.set_yticks(range(n_cpds)); ax1.set_yticklabels([n[:18] for n in names], fontsize=9)
    ax1.set_title("Risk Tier Heatmap (green=OK, orange=flag, red=concern)",
                  fontweight="bold", fontsize=11)
    for i in range(n_cpds):
        for j in range(len(endpoints_list)):
            val = int(risk_matrix[i,j])
            ax1.text(j, i, ["✓","!","✗"][val], ha="center", va="center",
                     fontsize=10, fontweight="bold", color="white")

    # ── 2. Animals avoided ────────────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 2])
    avoided = [results_panel[n]["dossier"].get("animals_avoided", 0) for n in names]
    bars = ax2.barh([n[:16] for n in names], avoided,
                    color=[colours[i%5] for i in range(n_cpds)], height=0.6)
    for bar, val in zip(bars, avoided):
        ax2.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
                 f"~{val}", va="center", fontsize=9, fontweight="bold")
    ax2.set_xlabel("Estimated animals avoided", fontsize=10)
    ax2.set_title("3Rs Impact per Compound", fontweight="bold")
    ax2.grid(True, alpha=0.3, axis="x")

    # ── 3. QED vs hERG IC50 scatter ───────────────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 0])
    for i, (name, state) in enumerate(results_panel.items()):
        admet = state["admet_results"].get("admet", {})
        herg  = state["cardiac_results"].get("herg", {})
        qed   = admet.get("QED", 0.5)
        ic50  = herg.get("hERG_IC50_uM", 1.0)
        ax3.scatter(qed, np.log10(ic50+0.001), s=120, color=colours[i%5],
                    zorder=5, label=name[:14])
        ax3.annotate(name[:8], (qed, np.log10(ic50+0.001)),
                     textcoords="offset points", xytext=(4,4), fontsize=7)
    ax3.axhline(np.log10(THRESHOLDS["hERG_flag_uM"]), color="red",
                linestyle="--", lw=1.5, alpha=0.7, label="hERG concern")
    ax3.set_xlabel("QED score"); ax3.set_ylabel("log₁₀(hERG IC50 µM)")
    ax3.set_title("Drug-likeness vs Cardiac Safety", fontweight="bold")
    ax3.legend(fontsize=7); ax3.grid(True, alpha=0.3)

    # ── 4. PBPK Cmax bar chart ────────────────────────────────────────────────
    ax4 = fig.add_subplot(gs[1, 1])
    pop_means = []
    pop_p5    = []
    pop_p95   = []
    for name, state in results_panel.items():
        ppk = state["admet_results"].get("pop_pbpk", {})
        pop_means.append(ppk.get("Cmax_mean_uM", 0))
        pop_p5.append(   ppk.get("Cmax_p5_uM",  0))
        pop_p95.append(  ppk.get("Cmax_p95_uM", 0))

    x = np.arange(n_cpds)
    ax4.bar(x, pop_means, color=[colours[i%5] for i in range(n_cpds)], alpha=0.8, width=0.6)
    ax4.errorbar(x, pop_means,
                 yerr=[np.array(pop_means)-np.array(pop_p5),
                        np.array(pop_p95)-np.array(pop_means)],
                 fmt="none", color="black", capsize=4, lw=1.5)
    ax4.set_xticks(x); ax4.set_xticklabels([n[:12] for n in names], rotation=30, ha="right", fontsize=8)
    ax4.set_ylabel("Cmax (µM)"); ax4.set_title("Population PBPK Cmax (mean ± 5-95%)", fontweight="bold")
    ax4.grid(True, alpha=0.3, axis="y")

    # ── 5. Genotox concordance ────────────────────────────────────────────────
    ax5 = fig.add_subplot(gs[1, 2])
    sa_calls   = [results_panel[n]["geno_results"].get("structural_alerts",{}).get("sa_call","N/A") for n in names]
    qsar_calls = [results_panel[n]["geno_results"].get("qsar_ames",{}).get("qsar_call","N/A") for n in names]
    for i, (name, sa, qsar) in enumerate(zip(names, sa_calls, qsar_calls)):
        sa_v   = 1 if sa=="POSITIVE" else 0
        qsar_v = 1 if qsar=="POSITIVE" else 0
        color  = "#E74C3C" if sa_v and qsar_v else "#27AE60" if not sa_v and not qsar_v else "#F39C12"
        ax5.scatter(sa_v + np.random.uniform(-0.08,0.08),
                    qsar_v + np.random.uniform(-0.08,0.08),
                    s=200, color=color, zorder=5)
        ax5.annotate(name[:10], (sa_v, qsar_v), textcoords="offset points",
                     xytext=(6, 4), fontsize=7)
    ax5.set_xticks([0,1]); ax5.set_xticklabels(["SA Neg","SA Pos"])
    ax5.set_yticks([0,1]); ax5.set_yticklabels(["QSAR Neg","QSAR Pos"])
    ax5.set_title("ICH M7 Concordance\n(SA vs QSAR)", fontweight="bold")
    ax5.set_xlim(-0.3,1.3); ax5.set_ylim(-0.3,1.3)
    ax5.grid(True, alpha=0.3)
    for txt, x_pos, y_pos in [("Concordant\nNeg","−0.15","−0.15"),
                                ("Concordant\nPos","0.85","0.85")]:
        pass  # labels implicit

    # ── 6. Concerns radar chart ───────────────────────────────────────────────
    ax6 = fig.add_subplot(gs[2, 0], polar=True)
    categories = ["Geno","Cardiac","DILI","Skin","Endocrine","RM"]
    N = len(categories)
    angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]

    for i, (name, state) in enumerate(list(results_panel.items())[:3]):
        values = [
            2 if state["geno_concern"]=="CONCERN"     else 1 if "INV" in state["geno_concern"] else 0,
            2 if state["herg_risk"]=="HIGH"            else 1 if state["herg_risk"]=="MEDIUM"    else 0,
            2 if state["dili_concern"]=="Most-concern" else 1 if "Less" in state["dili_concern"] else 0,
            2 if state["skin_hazard"]=="SENSITISER"    else 0,
            2 if state["ed_concern"]                   else 0,
            2 if state["rm_risk"]=="HIGH"              else 1 if state["rm_risk"]=="MODERATE"    else 0,
        ]
        values += [values[0]]
        ax6.plot(angles, values, "o-", lw=2, color=colours[i%5], label=name[:12])
        ax6.fill(angles, values, alpha=0.07, color=colours[i%5])

    ax6.set_xticks(angles[:-1]); ax6.set_xticklabels(categories, fontsize=9)
    ax6.set_yticks([0,1,2]); ax6.set_yticklabels(["OK","Flag","Concern"], fontsize=7)
    ax6.set_title("Risk Radar", fontweight="bold", pad=15)
    ax6.legend(loc="upper right", bbox_to_anchor=(1.3,1.1), fontsize=7)
    ax6.set_ylim(0, 2)

    # ── 7. 3Rs impact summary ─────────────────────────────────────────────────
    ax7 = fig.add_subplot(gs[2, 1:])
    ax7.axis("off")
    total_avoided = sum(d["dossier"].get("animals_avoided",0) for d in results_panel.values())
    summary_data  = [["Endpoint", "Method", "Guideline", "Animals/test", "Status"]]
    replacements  = [
        ["Ames mutagenicity",     "SA + QSAR (ICH M7)",  "ICH M7(R2) 2023",     "0 (bacteria)", "✓ REPLACED"],
        ["hERG / QT in vivo",    "CiPA + hiPS-CM MEA",  "ICH E14/S7B 2022",    "~10 rabbits",  "✓ REPLACED"],
        ["28-day hepatotox",      "Liver organoid+OoC",  "InSphero validated",  "~40 rats",     "✓ REPLACED"],
        ["LLNA skin sensitise",   "OECD TG 497 DA2",     "OECD TG 497 (2023)",  "~26 mice",     "✓ REPLACED"],
        ["LD50 acute oral",       "QSAR GHS predict",    "OECD GD 69",          "~10 rats",     "✓ REPLACED"],
        ["Endocrine (uterotrophic)","ER/AR in silico",   "OECD TG 455/457",     "~25 rats",     "✓ REPLACED"],
        ["Nephrotox study",       "Kidney OoC KIM-1",    "Mimetas validated",   "~20 rats",     "✓ REPLACED"],
    ]
    table = ax7.table(
        cellText=replacements,
        colLabels=["Endpoint","NAM Method","Guideline","Animals saved/test","Status"],
        cellLoc="center", loc="center", bbox=[0,0,1,1]
    )
    table.auto_set_font_size(False); table.set_fontsize(8.5)
    for j in range(5):
        table[0,j].set_facecolor("#1565C0")
        table[0,j].set_text_props(color="white", fontweight="bold")
    for i in range(1, len(replacements)+1):
        for j in range(5):
            bg = "#EEF9EE" if i%2==0 else "white"
            table[i,j].set_facecolor(bg)
        table[i,4].set_text_props(color="#27AE60", fontweight="bold")
    ax7.set_title(f"Animal-Free Replacement Matrix  |  Total ~{total_avoided} animals avoided this panel",
                  fontweight="bold", pad=12)

    fig.suptitle("ToxAgent Pro — Animal-Free Toxicology Dashboard",
                 fontsize=15, fontweight="bold", y=1.01)
    plt.savefig("toxagent_dashboard.png", dpi=130, bbox_inches="tight")
    plt.show()
    print("Dashboard saved: toxagent_dashboard.png")

plot_toxagent_dashboard(results_panel)

---
## Section 8 — Interactive ReAct Loop

Ask ToxAgent Pro a natural-language question about any compound. The agent reasons, selects tools, and builds an answer step by step.

In [ ]:
# ── 8.1 ReAct loop for interactive queries ────────────────────────────────────

class ToxAgentReAct:
    """
    ReAct agent for natural language toxicology queries.
    Thought → Action (tool) → Observation → repeat → Answer.
    """
    TOOL_ROUTING = {
        "cardiac":       ["herg","cipa","hipscm_mea"],
        "herg":          ["herg","cipa"],
        "genotox":       ["structural_alerts","qsar_ames","genotox_battery"],
        "mutagen":       ["structural_alerts","qsar_ames"],
        "liver":         ["dili","liver_organoid","ooc_liver"],
        "hepato":        ["dili","liver_organoid"],
        "kidney":        ["ooc_kidney"],
        "skin":          ["skin_sens"],
        "cns":           ["bbb","neurotox"],
        "endocrine":     ["endocrine"],
        "safe":          ["admet","pains","structural_alerts","qsar_ames","herg"],
        "admet":         ["admet","pbpk","pop_pbpk"],
        "dose":          ["pbpk","pop_pbpk"],
        "mechanism":     ["aop","reactive_met"],
        "full":          list(TOOLS.keys()),
        "all":           list(TOOLS.keys()),
    }

    def __init__(self, memory: ToxAgentMemory):
        self.memory = memory

    def route_tools(self, query: str) -> list[str]:
        q = query.lower()
        for keyword, tools in self.TOOL_ROUTING.items():
            if keyword in q:
                return tools
        return ["admet","herg","dili","structural_alerts"]  # default subset

    def run(self, smiles: str, query: str, max_steps: int = 5) -> str:
        print(f"\nQuery: {query}")
        print(f"SMILES: {smiles[:50]}")
        print("-" * 55)
        tool_list = self.route_tools(query)
        observations = {}
        for step, tool_name in enumerate(tool_list[:max_steps], 1):
            tool = TOOLS.get(tool_name)
            if not tool or tool.func is None:
                continue
            print(f"  Step {step}: Thought — running {tool.name}...")
            try:
                result = tool.func(smiles=smiles)
                obs    = result if isinstance(result, dict) else json.loads(result)
                observations[tool_name] = obs
                # Extract key value for reporting
                key_vals = {k:v for k,v in obs.items()
                            if isinstance(v,(str,float,int,bool)) and k!="replaces"}
                kv_str = "  ".join(f"{k}={v}" for k,v in list(key_vals.items())[:3])
                print(f"    Obs: {kv_str}")
            except Exception as e:
                print(f"    Error: {e}")

        # Build answer
        answer_parts = [f"\nAnswer for: {query}", f"Compound: {smiles[:40]}...", ""]
        for tool_name, obs in observations.items():
            tool = TOOLS.get(tool_name)
            if tool:
                key_findings = {k:v for k,v in obs.items()
                                if isinstance(v,(str,float,int,bool))
                                and k not in ("replaces","guideline","reference","note",
                                               "3Rs_saving","recommendation","action")}
                answer_parts.append(f"[{tool.name}]")
                for k, v in list(key_findings.items())[:4]:
                    answer_parts.append(f"  {k}: {v}")

        return "\n".join(answer_parts)

react_agent = ToxAgentReAct(memory)

# Run interactive queries
queries = [
    ("CC(=O)Oc1ccccc1C(=O)O",  "Is this compound safe for cardiac use?"),
    ("COCCNC(=O)c1cc(Cl)c(N)cc1OC1CCNCC1",  "What is the cardiac hERG risk?"),
    ("Cc1ccc(CC2SC(=O)NC2=O)cc1OCC(C)(C)c1ccc(O)cc1", "Does this cause liver toxicity?"),
    ("O=[N+]([O-])c1ccccc1", "Check genotoxicity and mutagenicity"),
]

for smi, query in queries:
    answer = react_agent.run(smi, query, max_steps=3)
    print(answer)
    print()

---
## Section 9 — Regulatory Dossier Output

In [ ]:
# ── 9.1 Generate complete regulatory dossier ─────────────────────────────────
def generate_full_dossier(results_panel: dict) -> pd.DataFrame:
    """
    Generate a pandas DataFrame summary of the full panel assessment.
    Formatted for regulatory submission per OECD GD 255.
    """
    rows = []
    for name, state in results_panel.items():
        dos   = state.get("dossier", {})
        admet = state["admet_results"].get("admet", {})
        pbpk  = state["admet_results"].get("pbpk", {})
        herg  = state["cardiac_results"].get("herg", {})
        mea   = state["cardiac_results"].get("hipscm_mea", {})
        dili  = state["dili_results"].get("dili", {})
        org   = state["dili_results"].get("liver_organoid", {})
        skin  = state["organ_results"].get("skin_sens", {})
        geno  = state["geno_results"].get("genotox_battery", {})
        uq    = compute_uncertainty_profile(state)
        rows.append({
            "Compound":           name,
            "MW":                 admet.get("MW",""),
            "LogP":               admet.get("LogP",""),
            "QED":                admet.get("QED",""),
            "Ro5_viols":          admet.get("Ro5_violations",""),
            "SA_call":            state["geno_results"].get("structural_alerts",{}).get("sa_call",""),
            "QSAR_Ames":          state["geno_results"].get("qsar_ames",{}).get("qsar_call",""),
            "Geno_ICH_M7":        geno.get("overall",""),
            "hERG_IC50_uM":       herg.get("hERG_IC50_uM",""),
            "hERG_risk":          state["herg_risk"],
            "CiPA_TdP":           state["cipa_tdp"],
            "dFPDc_ms":           mea.get("delta_FPDc_ms",""),
            "DILIrank":           state["dili_concern"],
            "Organoid_ATP_pct":   org.get("ATP_pct",""),
            "Skin_sens":          state["skin_hazard"],
            "ED_concern":         state["ed_concern"],
            "AED_mg_kg_d":        pbpk.get("AED_mg_kg_day",""),
            "Conclusion":         dos.get("conclusion",""),
            "Confidence":         dos.get("confidence",""),
            "UQ_confidence":      uq["overall_confidence"],
            "Animals_avoided":    dos.get("animals_avoided",0),
            "HITL":               state.get("hitl_decision","N/A"),
            "n_concerns":         dos.get("n_concerns",0),
        })

    df = pd.DataFrame(rows)
    print("REGULATORY DOSSIER SUMMARY (OECD GD 255 IATA Format)")
    print("=" * 80)
    print(df[["Compound","Geno_ICH_M7","hERG_risk","DILIrank","Skin_sens",
              "Conclusion","Animals_avoided"]].to_string(index=False))
    print(f"\nTotal animals avoided across panel: ~{df['Animals_avoided'].sum()}")
    print(f"\n3Rs Legal basis: FDA Modernization Act 2.0 (2022) — animal data")
    print("no longer required for IND applications where NAM data is available.")
    return df

dossier_df = generate_full_dossier(results_panel)
dossier_df.to_csv("toxagent_dossier.csv", index=False)
print("\nDossier saved: toxagent_dossier.csv")

In [ ]:
# ── 9.2 Print individual full reports ─────────────────────────────────────────
print("Individual reports:")
for name, state in list(results_panel.items())[:2]:
    print(state["final_report"])

In [ ]:
# ── 9.3 ToxAgent Pro cheatsheet ──────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║              TOXAGENT PRO — Quick Reference                             ║
╠══════════════════════════════════════════════════════════════════════════╣
║ RUN THE FULL PIPELINE                                                    ║
║  supervisor = SupervisorAgent()                                         ║
║  state = supervisor.run_full_pipeline(smiles, name, intended_use)       ║
║  print(state['final_report'])                                           ║
╠══════════════════════════════════════════════════════════════════════════╣
║ RUN A SINGLE TOOL                                                        ║
║  TOOLS['herg'].func(smiles='CC(=O)Oc1ccccc1C(=O)O')                   ║
║  TOOLS['structural_alerts'].func(smiles=smi)                            ║
║  TOOLS['liver_organoid'].func(smiles=smi, concentration_uM=10.0)        ║
╠══════════════════════════════════════════════════════════════════════════╣
║ RUN A SPECIALIST AGENT                                                   ║
║  results = AGENTS['cardiac_agent'].run(smiles)  # hERG+CiPA+MEA        ║
║  results = AGENTS['dili_agent'].run(smiles)     # DILI+organoid+OoC    ║
╠══════════════════════════════════════════════════════════════════════════╣
║ INTERACTIVE QUERY                                                        ║
║  react = ToxAgentReAct(memory)                                          ║
║  answer = react.run(smiles, 'Is this compound hepatotoxic?')            ║
╠══════════════════════════════════════════════════════════════════════════╣
║ 25 TOOLS SUMMARY                                                         ║
║  admet · structural_alerts · qsar_ames · qsar_ld50 · herg               ║
║  cipa · hipscm_mea · dili · liver_organoid · ooc_liver                  ║
║  ooc_kidney · skin_sens · bbb · neurotox · pbpk                        ║
║  pop_pbpk · cytotox_corr · pains · genotox_battery · reactive_met      ║
║  endocrine · aop · read_across · literature · regulatory               ║
╠══════════════════════════════════════════════════════════════════════════╣
║ REGULATORY ALIGNMENT                                                     ║
║  ICH M7(R2) 2023     Genotoxicity (SA + QSAR)                          ║
║  ICH E14/S7B 2022    CiPA multi-channel + hiPS-CM MEA                  ║
║  OECD TG 497 (2023)  Skin sensitisation DA2 (2o3)                      ║
║  EPA HTTK            IVIVE (fup + CLint → AED → TTC)                   ║
║  OECD GD 255         IATA weight-of-evidence framework                  ║
║  FDA FMA 2.0 (2022)  Animal data no longer required for IND             ║
╠══════════════════════════════════════════════════════════════════════════╣
║ 3Rs IMPACT PER RUN (typical pharmaceutical compound)                    ║
║  Ames + MNT + CA battery:    ~50 animals avoided                       ║
║  hERG + in vivo QT study:    ~10 rabbits avoided                       ║
║  28-day hepatotoxicity:      ~40 rats avoided                           ║
║  LLNA skin sensitisation:    ~26 mice avoided                           ║
║  LD50 acute oral:            ~10 rats avoided                           ║
║  ─────────────────────────────────────────────────                     ║
║  Total per compound:         ~130 animals avoided                       ║
╚══════════════════════════════════════════════════════════════════════════╝
""")